## IMAGE INFERENCE

In [ ]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv
from PIL import Image
import io

# Load environment variables from .env file
load_dotenv()

def process_image_with_nova_arn_simple(
    image_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this image content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process image using AWS Bedrock Nova Pro with ARN profile - Simplified version
    Uses only the working invoke_model method
    
    Args:
        image_path: Path to the image file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for image analysis
        aws_profile: Optional AWS profile name
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Image file validation
        if not os.path.exists(image_path):
            return f"Error: Image file not found: {image_path}"
        
        file_size = os.path.getsize(image_path)
        image_name = os.path.basename(image_path)
        
        print(f"Processing image: {image_name}")
        print(f"File size: {file_size / 1024:.2f} KB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Image too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Validate and potentially fix image using PIL
        try:
            with Image.open(image_path) as img:
                print(f"Original image format: {img.format}")
                print(f"Original image mode: {img.mode}")
                print(f"Original image size: {img.size}")
                
                # Convert to RGB if necessary (Nova Pro works best with RGB)
                if img.mode != 'RGB':
                    print(f"Converting from {img.mode} to RGB")
                    img = img.convert('RGB')
                
                # Resize if too large (Nova Pro has limits)
                max_size = 2048
                if max(img.size) > max_size:
                    print(f"Resizing image from {img.size} to fit {max_size}px limit")
                    img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
                
                # Save to bytes buffer in PNG format for consistency
                buffer = io.BytesIO()
                img.save(buffer, format='PNG')
                image_bytes = buffer.getvalue()
                
                print(f"Processed image bytes length: {len(image_bytes)}")
                print(f"Final image format: PNG")
                print(f"Final image size: {img.size}")
                
        except Exception as e:
            print(f"PIL processing failed: {str(e)}")
            # Fallback to original file
            with open(image_path, 'rb') as image_file:
                image_bytes = image_file.read()
        
        # Encode to base64
        print("Encoding image to base64...")
        image_b64 = base64.b64encode(image_bytes).decode('utf-8')
        
        print(f"Base64 string length: {len(image_b64)}")
        
        # Prepare the message content - use PNG format for consistency
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "png",  # Always use PNG for consistency
                            "source": {
                                "bytes": image_b64
                            }
                        }
                    },
                    {"text": prompt}
                ]
            }
        ]
        
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Use only the working invoke_model method
        response = bedrock.invoke_model(
            modelId="amazon.nova-pro-v1:0",
            body=json.dumps({
                "messages": messages,
                "inferenceConfig": inference_config
            }),
            contentType="application/json"
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        result = response_body['output']['message']['content'][0]['text']
        print("Image analysis completed successfully!")
        return result
        
    except FileNotFoundError:
        return f"Error: Image file not found: {image_path}"
    except Exception as e:
        return f"Error processing image: {str(e)}"

def validate_arn_format(arn: str) -> bool:
    """Validate ARN format"""
    return arn.startswith("arn:aws:bedrock:") and ("inference-profile" in arn or "model-access-policy" in arn)

# Test the fixed image processing
if __name__ == "__main__":
    # Configuration

    # ------->>>>>> change the profile arn to the one you want to use
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    
    #----------------->>>>>>>>>> Test with the new stock chart image
    image_file = "data/images/stock_chart.png"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    print("="*60)
    print("TESTING SIMPLIFIED IMAGE PROCESSING")
    print("="*60)
    
    if os.path.exists(image_file):
        print(f"Testing with: {image_file}")
        image_result = process_image_with_nova_arn_simple(
            image_file,
            profile_arn,
            "Analyze this financial chart image and describe what you see. Focus on trends, patterns, and any notable data points."
        )
        print(f"\nImage Analysis Result:")
        print("-" * 50)
        print(image_result)
    else:
        print(f"Image file not found: {image_file}")
        
        # Try with the fixed red square
        fallback_image = "data/images/test_red_square_fixed.png"
        if os.path.exists(fallback_image):
            print(f"Testing with fallback: {fallback_image}")
            image_result = process_image_with_nova_arn_simple(
                fallback_image,
                profile_arn,
                "Describe this simple image."
            )
            print(f"\nImage Analysis Result:")
            print("-" * 50)
            print(image_result)
        else:
            print("No suitable test images found")


TESTING SIMPLIFIED IMAGE PROCESSING
Testing with: data/images/stock_chart.png
Processing image: stock_chart.png
File size: 50.98 KB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0
Original image format: PNG
Original image mode: RGBA
Original image size: (1182, 880)
Converting from RGBA to RGB
Processed image bytes length: 49304
Final image format: PNG
Final image size: (1182, 880)
Encoding image to base64...
Base64 string length: 65740
Sending request to Bedrock Nova Pro...
Image analysis completed successfully!

Image Analysis Result:
--------------------------------------------------
The chart depicts a sample stock price over five time periods. Initially, the stock price starts at $100 in the first period, increases sharply to $120 in the second period, then drops to $110 in the third period. Subsequently, the price rises dramatically to $140 in the fourth period before declining to $130 in the fifth period. This pattern sho

## VIDEO INFERENCE

In [ ]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

def process_video_with_nova_arn_simple(
    video_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this video content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process video using AWS Bedrock Nova Pro with ARN profile - Simplified version
    Uses only the working invoke_model method
    
    Args:
        video_path: Path to the video file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for video analysis
        aws_profile: Optional AWS profile name (instead of hardcoded keys)
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Video file validation
        if not os.path.exists(video_path):
            return f"Error: Video file not found: {video_path}"
        
        file_size = os.path.getsize(video_path)
        video_name = os.path.basename(video_path)
        
        print(f"Processing video: {video_name}")
        print(f"File size: {file_size / (1024*1024):.2f} MB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Video too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Read and encode video file
        print("Encoding video to base64...")
        with open(video_path, 'rb') as video_file:
            video_bytes = video_file.read()
            video_b64 = base64.b64encode(video_bytes).decode('utf-8')
        
        print(f"Video bytes length: {len(video_bytes)}")
        print(f"Base64 string length: {len(video_b64)}")
        print(f"Base64 starts with: {video_b64[:50]}...")
        
        # Determine video format from file extension
        file_ext = os.path.splitext(video_path)[1].lower()
        format_map = {
            '.mp4': 'mp4',
            '.mov': 'mov',
            '.avi': 'avi',
            '.webm': 'webm'
        }
        video_format = format_map.get(file_ext, 'mp4')
        
        print(f"Detected video format: {video_format}")
        print(f"File extension: {file_ext}")
        
        # Prepare the message content
        messages = [
        {
            "role": "user",
            "content": [
                {
                    "video": {
                        "format": video_format,  # Use the detected format
                        "source": {
                            "bytes": video_b64
                        }
                    }
                },
                {"text": prompt}
            ]
        }
    ]
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Use only the working invoke_model method
        response = bedrock.invoke_model(
            modelId="amazon.nova-pro-v1:0",
            body=json.dumps({
                "messages": messages,
                "inferenceConfig": inference_config
            }),
            contentType="application/json"
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        result = response_body['output']['message']['content'][0]['text']
        print("Video analysis completed successfully!")
        return result
        
    except FileNotFoundError:
        return f"Error: Video file not found: {video_path}"
    except Exception as e:
        return f"Error processing video: {str(e)}"



# Usage example
if __name__ == "__main__":
    # Configuration
    # ------->>>>>> change the profile arn to the one you want to use
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    video_file = "data/videos/appleq1.mp4"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    # Process video
    if os.path.exists(video_file):
        result = process_video_with_nova_arn_simple(
            video_file, 
            profile_arn,
            "Analyze this video and provide key insights about the content, actions, and any notable elements."
        )
        print(f"\nVideo Analysis Result:")
        print("-" * 50)
        print(result)
    else:
        print(f"Video file not found: {video_file}")
    
    # Test image processing
    print("\n" + "="*60)
    print("TESTING IMAGE PROCESSING")
    print("="*60)
    
    

Processing video: appleq1.mp4
File size: 3.71 MB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0
Encoding video to base64...
Video bytes length: 3885780
Base64 string length: 5181040
Base64 starts with: AAAAIGZ0eXBpc29tAAACAGlzb21pc28yYXZjMW1wNDEAAvS9bW...
Detected video format: mp4
File extension: .mp4
Sending request to Bedrock Nova Pro...
Video analysis completed successfully!

Video Analysis Result:
--------------------------------------------------
The video depicts a financial trading dashboard displaying stock market data for Apple Inc. (AAPL) and the NASDAQ-100 Index (INTC). The dashboard shows historical stock prices, volume, and technical indicators such as moving averages and relative strength index (RSI). The data suggests a positive trend for both stocks, with Apple's stock price increasing significantly over the past year. The dashboard also includes a news headline about Apple's financial results for the 2023 Q4 

## AUDIO-ANALYSIS

In [ ]:
import os
import asyncio
import base64
import json
import uuid
import pyaudio
from aws_sdk_bedrock_runtime.client import BedrockRuntimeClient, InvokeModelWithBidirectionalStreamOperationInput
from aws_sdk_bedrock_runtime.models import InvokeModelWithBidirectionalStreamInputChunk, BidirectionalInputPayloadPart
from aws_sdk_bedrock_runtime.config import Config, HTTPAuthSchemeResolver, SigV4AuthScheme
from smithy_aws_core.credentials_resolvers.environment import EnvironmentCredentialsResolver

from dotenv import load_dotenv
load_dotenv()

# Audio configuration
INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000
CHANNELS = 1
FORMAT = pyaudio.paInt16
CHUNK_SIZE = 1024

class SimpleNovaSonic:
    def __init__(self, model_id='amazon.nova-sonic-v1:0', region='us-east-1'):
        self.model_id = model_id
        self.region = region
        self.client = None
        self.stream = None
        self.response = None
        self.is_active = False
        self.prompt_name = str(uuid.uuid4())
        self.content_name = str(uuid.uuid4())
        self.audio_content_name = str(uuid.uuid4())
        self.audio_queue = asyncio.Queue()
        self.role = None
        self.display_assistant_text = False
        
    def _initialize_client(self):
        """Initialize the Bedrock client."""
        config = Config(
            endpoint_uri=f"https://bedrock-runtime.{self.region}.amazonaws.com",
            region=self.region,
            aws_credentials_identity_resolver=EnvironmentCredentialsResolver(),
            http_auth_scheme_resolver=HTTPAuthSchemeResolver(),
            http_auth_schemes={"aws.auth#sigv4": SigV4AuthScheme()}
        )
        self.client = BedrockRuntimeClient(config=config)
    
    async def send_event(self, event_json):
        """Send an event to the stream."""
        event = InvokeModelWithBidirectionalStreamInputChunk(
            value=BidirectionalInputPayloadPart(bytes_=event_json.encode('utf-8'))
        )
        await self.stream.input_stream.send(event)
    
    async def start_session(self):
        """Start a new session with Nova Sonic."""
        if not self.client:
            self._initialize_client()
            
        # Initialize the stream
        self.stream = await self.client.invoke_model_with_bidirectional_stream(
            InvokeModelWithBidirectionalStreamOperationInput(model_id=self.model_id)
        )
        self.is_active = True
        
        # Send session start event
        session_start = '''
        {
          "event": {
            "sessionStart": {
              "inferenceConfiguration": {
                "maxTokens": 1024,
                "topP": 0.9,
                "temperature": 0.1
              }
            }
          }
        }
        '''
        await self.send_event(session_start)
        
        # Send prompt start event
        prompt_start = f'''
        {{
          "event": {{
            "promptStart": {{
              "promptName": "{self.prompt_name}",
              "textOutputConfiguration": {{
                "mediaType": "text/plain"
              }},
              "audioOutputConfiguration": {{
                "mediaType": "audio/lpcm",
                "sampleRateHertz": 24000,
                "sampleSizeBits": 16,
                "channelCount": 1,
                "voiceId": "matthew",
                "encoding": "base64",
                "audioType": "SPEECH"
              }}
            }}
          }}
        }}
        '''
        await self.send_event(prompt_start)
        
        # Send system prompt
        text_content_start = f'''
        {{
            "event": {{
                "contentStart": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.content_name}",
                    "type": "TEXT",
                    "interactive": false,
                    "role": "SYSTEM",
                    "textInputConfiguration": {{
                        "mediaType": "text/plain"
                    }}
                }}
            }}
        }}
        '''
        await self.send_event(text_content_start)
        
        system_prompt = "You are a friendly assistant. The user and you will engage in a spoken dialog " \
            "exchanging the transcripts of a natural real-time conversation. Keep your responses short, " \
            "generally two or three sentences for chatty scenarios."
        


        text_input = f'''
        {{
            "event": {{
                "textInput": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.content_name}",
                    "content": "{system_prompt}"
                }}
            }}
        }}
        '''
        await self.send_event(text_input)
        
        text_content_end = f'''
        {{
            "event": {{
                "contentEnd": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.content_name}"
                }}
            }}
        }}
        '''
        await self.send_event(text_content_end)
        
        # Start processing responses
        self.response = asyncio.create_task(self._process_responses())
    
    async def start_audio_input(self):
        """Start audio input stream."""
        audio_content_start = f'''
        {{
            "event": {{
                "contentStart": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.audio_content_name}",
                    "type": "AUDIO",
                    "interactive": true,
                    "role": "USER",
                    "audioInputConfiguration": {{
                        "mediaType": "audio/lpcm",
                        "sampleRateHertz": 16000,
                        "sampleSizeBits": 16,
                        "channelCount": 1,
                        "audioType": "SPEECH",
                        "encoding": "base64"
                    }}
                }}
            }}
        }}
        '''
        await self.send_event(audio_content_start)
    
    async def send_audio_chunk(self, audio_bytes):
        """Send an audio chunk to the stream."""
        if not self.is_active:
            return
            
        blob = base64.b64encode(audio_bytes)
        audio_event = f'''
        {{
            "event": {{
                "audioInput": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.audio_content_name}",
                    "content": "{blob.decode('utf-8')}"
                }}
            }}
        }}
        '''
        await self.send_event(audio_event)
    
    async def end_audio_input(self):
        """End audio input stream."""
        audio_content_end = f'''
        {{
            "event": {{
                "contentEnd": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.audio_content_name}"
                }}
            }}
        }}
        '''
        await self.send_event(audio_content_end)
    
    async def end_session(self):
        """End the session."""
        if not self.is_active:
            return
            
        prompt_end = f'''
        {{
            "event": {{
                "promptEnd": {{
                    "promptName": "{self.prompt_name}"
                }}
            }}
        }}
        '''
        await self.send_event(prompt_end)
        
        session_end = '''
        {
            "event": {
                "sessionEnd": {}
            }
        }
        '''
        await self.send_event(session_end)
        # close the stream
        await self.stream.input_stream.close()
    
    async def _process_responses(self):
        """Process responses from the stream."""
        try:
            while self.is_active:
                output = await self.stream.await_output()
                result = await output[1].receive()
                
                if result.value and result.value.bytes_:
                    response_data = result.value.bytes_.decode('utf-8')
                    json_data = json.loads(response_data)
                    
                    if 'event' in json_data:
                        # Handle content start event
                        if 'contentStart' in json_data['event']:
                            content_start = json_data['event']['contentStart'] 
                            # set role
                            self.role = content_start['role']
                            # Check for speculative content
                            if 'additionalModelFields' in content_start:
                                additional_fields = json.loads(content_start['additionalModelFields'])
                                if additional_fields.get('generationStage') == 'SPECULATIVE':
                                    self.display_assistant_text = True
                                else:
                                    self.display_assistant_text = False
                                
                        # Handle text output event
                        elif 'textOutput' in json_data['event']:
                            text = json_data['event']['textOutput']['content']    
                           
                            if (self.role == "ASSISTANT" and self.display_assistant_text):
                                print(f"Assistant: {text}")
                            elif self.role == "USER":
                                print(f"User: {text}")
                        
                        # Handle audio output
                        elif 'audioOutput' in json_data['event']:
                            audio_content = json_data['event']['audioOutput']['content']
                            audio_bytes = base64.b64decode(audio_content)
                            await self.audio_queue.put(audio_bytes)
        except Exception as e:
            print(f"Error processing responses: {e}")
    
    async def play_audio(self):
        """Play audio responses."""
        p = pyaudio.PyAudio()
        stream = p.open(
            format=FORMAT,
            channels=CHANNELS,
            rate=OUTPUT_SAMPLE_RATE,
            output=True
        )
        
        try:
            while self.is_active:
                audio_data = await self.audio_queue.get()
                stream.write(audio_data)
        except Exception as e:
            print(f"Error playing audio: {e}")
        finally:
            stream.stop_stream()
            stream.close()
            p.terminate()
            print("Audio playing stopped.")

    async def capture_audio(self):
        """Capture audio from microphone and send to Nova Sonic."""
        p = pyaudio.PyAudio()
        stream = p.open(
            format=FORMAT,
            channels=CHANNELS,
            rate=INPUT_SAMPLE_RATE,
            input=True,
            frames_per_buffer=CHUNK_SIZE
        )
        
        print("Starting audio capture. Speak into your microphone...")
        print("Press Enter to stop...")
        
        await self.start_audio_input()
        
        try:
            while self.is_active:
                audio_data = stream.read(CHUNK_SIZE, exception_on_overflow=False)
                await self.send_audio_chunk(audio_data)
                await asyncio.sleep(0.01)
        except Exception as e:
            print(f"Error capturing audio: {e}")
        finally:
            stream.stop_stream()
            stream.close()
            p.terminate()
            print("Audio capture stopped.")
            await self.end_audio_input()

async def main():
    # Create Nova Sonic client
    nova_client = SimpleNovaSonic()
    
    # Start session
    await nova_client.start_session()
    
    # Start audio playback task
    playback_task = asyncio.create_task(nova_client.play_audio())
    
    # Start audio capture task
    capture_task = asyncio.create_task(nova_client.capture_audio())
    
    # Wait for user to press Enter to stop
    await asyncio.get_event_loop().run_in_executor(None, input)
        
    # First cancel the tasks
    tasks = []
    if not playback_task.done():
        tasks.append(playback_task)
    if not capture_task.done():
        tasks.append(capture_task)
    for task in tasks:
        task.cancel()
    if tasks:
        await asyncio.gather(*tasks, return_exceptions=True)
    
    # End session
    await nova_client.end_session()
    nova_client.is_active = False

    # cancel the response task
    if nova_client.response and not nova_client.response.done():
        nova_client.response.cancel()

    print("Session ended")

if __name__ == "__main__":
    # Set AWS credentials if not using environment variables
    # os.environ['AWS_ACCESS_KEY_ID'] = "your-access-key"
    # os.environ['AWS_SECRET_ACCESS_KEY'] = "your-secret-key"
    # os.environ['AWS_DEFAULT_REGION'] = "us-east-1"

    asyncio.run(main())

TESTING SIMPLIFIED AUDIO PROCESSING
Testing with: data/audio/test_tone.wav
Processing audio: test_tone.wav
File size: 31.29 KB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-sonic-v1:0
Encoding audio to base64...
Audio bytes length: 32044
Base64 string length: 42728
Detected audio format: wav
  AUDIO PROCESSING LIMITATION:
Nova Sonic requires AWS SDK v2 and bidirectional streaming.
The standard boto3 Bedrock API doesn't support Nova Sonic.

 RECOMMENDATION:
For now, use Image and Video analysis which work perfectly!
 Image Analysis: process_image_with_nova_arn_simple()
 Video Analysis: process_video_with_nova_arn_simple()
⏸  Audio Analysis: Requires AWS SDK v2 (not available)

 To enable audio processing later:
1. Install AWS SDK v2: pip install aws-sdk-bedrock-runtime
2. Use the standalone script: simple_nova_sonic.py

Audio Analysis Result:
--------------------------------------------------
Audio processing requires AWS SDK v2. Use I